# Tutorial: TabularPathExtractor

This notebook shows how to use `TabularPathExtractor` to convert tabular data into graph-ready entities and relations.

It focuses on extractor behavior only. Graph database initialization is intentionally skipped.

## What the extractor does

For each table row, the extractor creates one main entity identified by `table_name + primary_key`.

- Primary-key columns become part of the main entity identity and properties.
- Explicit foreign keys create edges to row entities in other tables.
- Scalar columns such as integers, floats, booleans, and ISO-like dates become properties by default.
- Text columns become related value entities by default.
- Explicit `RelationColumnSpec(..., split_separator=...)` lets one denormalized cell create multiple related entities.

This tutorial is intentionally pandas-first: all examples use in-memory DataFrames and no filesystem writes.

In [1]:
import autoroot  # noqa

from pprint import pprint

import pandas as pd

from ogre_kg.kg_processors import (
    ForeignKeySpec,
    RelationColumnSpec,
    TabularPathExtractor,
    TabularTableSpec,
)

All examples below operate directly on pandas DataFrames.

In [2]:
def run_extractor(*tables: TabularTableSpec):
    extractor = TabularPathExtractor(tables=list(tables))
    documents = extractor.load_documents()
    transformed_nodes = extractor(documents)
    return documents, transformed_nodes


def graph_view(node):
    return {
        "text": node.text,
        "entities": [
            {
                "name": entity.name,
                "label": entity.label,
                "properties": entity.properties,
            }
            for entity in node.metadata["nodes"]
        ],
        "relations": [
            {
                "label": relation.label,
                "source_id": relation.source_id,
                "target_id": relation.target_id,
                "properties": relation.properties,
            }
            for relation in node.metadata["relations"]
        ],
    }

## 1. Single-table extraction from a pandas DataFrame

Here `revenue_usd` is numeric, so it becomes a property on the main company entity. `industry` is text, so it becomes a related entity.

In [3]:
companies_df = pd.DataFrame(
    [
        {"company_id": "MSFT", "industry": "Tech", "revenue_usd": 123456789},
        {"company_id": "AAPL", "industry": "Consumer Electronics", "revenue_usd": 42},
    ]
)

documents, transformed_nodes = run_extractor(
    TabularTableSpec(
        name="companies",
        primary_key="company_id",
        dataframe=companies_df,
    )
)

pprint(graph_view(transformed_nodes[0]))

{'entities': [{'label': 'Companies',
               'name': 'companies:company_id=MSFT',
               'properties': {'company_id': 'MSFT', 'revenue_usd': 123456789}},
              {'label': 'Industry',
               'name': 'Industry:Tech',
               'properties': {'source_column': 'industry', 'value': 'Tech'}}],
 'relations': [{'label': 'HAS_INDUSTRY',
                'properties': {'source_column': 'industry'},
                'source_id': 'companies:company_id=MSFT',
                'target_id': 'Industry:Tech'}],
 'text': 'Table: companies\n'
         'company_id: MSFT\n'
         'industry: Tech\n'
         'revenue_usd: 123456789'}


## 2. Explicit property columns

Some values are textual but still should remain properties on the main entity. A good example is a human-readable revenue string such as `"123 USD"`.

In [4]:
company_metrics_df = pd.DataFrame(
    [
        {"company_id": "MSFT", "revenue_text": "123 USD", "headquarters": "Redmond"},
    ]
)

documents, transformed_nodes = run_extractor(
    TabularTableSpec(
        name="company_metrics",
        primary_key="company_id",
        dataframe=company_metrics_df,
        property_columns=["revenue_text"],
    )
)

pprint(graph_view(transformed_nodes[0]))

{'entities': [{'label': 'CompanyMetrics',
               'name': 'company_metrics:company_id=MSFT',
               'properties': {'company_id': 'MSFT', 'revenue_text': '123 USD'}},
              {'label': 'Headquarters',
               'name': 'Headquarters:Redmond',
               'properties': {'source_column': 'headquarters',
                              'value': 'Redmond'}}],
 'relations': [{'label': 'HAS_HEADQUARTERS',
                'properties': {'source_column': 'headquarters'},
                'source_id': 'company_metrics:company_id=MSFT',
                'target_id': 'Headquarters:Redmond'}],
 'text': 'Table: company_metrics\n'
         'company_id: MSFT\n'
         'revenue_text: 123 USD\n'
         'headquarters: Redmond'}


## 3. Multi-table extraction with explicit foreign keys

The employee row below creates:

- one main `Employees` entity
- one `BELONGS_TO_DEPARTMENT` edge to the matching department row entity
- one `HAS_TITLE` edge to a value entity built from the text column `title`

In [5]:
departments_df = pd.DataFrame(
    [
        {"dept_id": "D10", "name": "Engineering"},
        {"dept_id": "D20", "name": "Sales"},
    ]
)

employees_df = pd.DataFrame(
    [
        {"emp_id": "E1", "dept_id": "D10", "title": "Engineer"},
        {"emp_id": "E2", "dept_id": "D20", "title": "Account Executive"},
    ]
)

documents, transformed_nodes = run_extractor(
    TabularTableSpec(
        name="departments",
        primary_key="dept_id",
        dataframe=departments_df,
    ),
    TabularTableSpec(
        name="employees",
        primary_key="emp_id",
        dataframe=employees_df,
        foreign_keys=[
            ForeignKeySpec(
                source_columns="dept_id",
                target_table="departments",
                target_columns="dept_id",
                relation_label="BELONGS_TO_DEPARTMENT",
            )
        ],
    ),
)

employee_nodes = [
    node for node in transformed_nodes if node.metadata["ogre_kg_table_name"] == "employees"
]
pprint(graph_view(employee_nodes[0]))

{'entities': [{'label': 'Employees',
               'name': 'employees:emp_id=E1',
               'properties': {'emp_id': 'E1'}},
              {'label': 'Departments',
               'name': 'departments:dept_id=D10',
               'properties': {'dept_id': 'D10'}},
              {'label': 'Title',
               'name': 'Title:Engineer',
               'properties': {'source_column': 'title', 'value': 'Engineer'}}],
 'relations': [{'label': 'BELONGS_TO_DEPARTMENT',
                'properties': {'source_columns': ['dept_id'],
                               'target_columns': ['dept_id'],
                               'target_table': 'departments'},
                'source_id': 'employees:emp_id=E1',
                'target_id': 'departments:dept_id=D10'},
               {'label': 'HAS_TITLE',
                'properties': {'source_column': 'title'},
                'source_id': 'employees:emp_id=E1',
                'target_id': 'Title:Engineer'}],
 'text': 'Table: employees\nemp_i

## 4. Denormalized list-valued columns in a DataFrame

A pandas DataFrame can still contain one cell with multiple logical values, for example `"R&D, Sales"`. The extractor does not auto-split this by default, but it can do so when the column is explicitly declared with `split_separator=","`.

In [6]:
denormalized_companies_df = pd.DataFrame(
    [
        {"company_id": 1, "industry": "Tech", "departments": "R&D, Sales"},
        {"company_id": 2, "industry": "Finance", "departments": "Accounting, HR"},
    ]
)

documents, transformed_nodes = run_extractor(
    TabularTableSpec(
        name="denormalized_companies",
        primary_key="company_id",
        dataframe=denormalized_companies_df,
        relation_columns=[
            RelationColumnSpec(
                column="departments",
                relation_label="HAS_DEPARTMENT",
                target_label="Department",
                split_separator=",",
            )
        ],
    )
)

pprint(graph_view(transformed_nodes[0]))

{'entities': [{'label': 'DenormalizedCompanies',
               'name': 'denormalized_companies:company_id=1',
               'properties': {'company_id': 1}},
              {'label': 'Industry',
               'name': 'Industry:Tech',
               'properties': {'source_column': 'industry', 'value': 'Tech'}},
              {'label': 'Department',
               'name': 'Department:R&D',
               'properties': {'source_column': 'departments', 'value': 'R&D'}},
              {'label': 'Department',
               'name': 'Department:Sales',
               'properties': {'source_column': 'departments',
                              'value': 'Sales'}}],
 'relations': [{'label': 'HAS_INDUSTRY',
                'properties': {'source_column': 'industry'},
                'source_id': 'denormalized_companies:company_id=1',
                'target_id': 'Industry:Tech'},
               {'label': 'HAS_DEPARTMENT',
                'properties': {'source_column': 'departments'},
         

## 5. Pandas-native normalization before extraction

Sometimes you may want to normalize a denormalized table yourself with pandas before passing it to the extractor. For example, you can split and `explode()` a list-valued column into one row per value.

In [7]:
exploded_companies_df = denormalized_companies_df.copy()
exploded_companies_df["departments"] = exploded_companies_df["departments"].str.split(",")
exploded_companies_df = exploded_companies_df.explode("departments").reset_index(drop=True)
exploded_companies_df["departments"] = exploded_companies_df["departments"].str.strip()
exploded_companies_df = exploded_companies_df.reset_index(names="row_id")

exploded_companies_df

,row_id,company_id,industry,departments
0,0,1,Tech,R&D
1,1,1,Tech,Sales
2,2,2,Finance,Accounting
3,3,2,Finance,HR


After exploding the DataFrame yourself, the row grain changes. This example assigns a fresh `row_id` after `explode()` so each normalized row has a unique primary key while `departments` stays a regular single-valued relation column.

In [8]:
documents, transformed_nodes = run_extractor(
    TabularTableSpec(
        name="exploded_company_departments",
        primary_key="row_id",
        dataframe=exploded_companies_df,
        property_columns=["company_id", "industry"],
        relation_columns=[
            RelationColumnSpec(
                column="departments",
                relation_label="HAS_DEPARTMENT",
                target_label="Department",
            )
        ],
    )
)

pprint(graph_view(transformed_nodes[0]))

{'entities': [{'label': 'ExplodedCompanyDepartments',
               'name': 'exploded_company_departments:row_id=0',
               'properties': {'company_id': 1,
                              'industry': 'Tech',
                              'row_id': 0}},
              {'label': 'Department',
               'name': 'Department:R&D',
               'properties': {'source_column': 'departments', 'value': 'R&D'}}],
 'relations': [{'label': 'HAS_DEPARTMENT',
                'properties': {'source_column': 'departments'},
                'source_id': 'exploded_company_departments:row_id=0',
                'target_id': 'Department:R&D'}],
 'text': 'Table: exploded_company_departments\n'
         'row_id: 0\n'
         'company_id: 1\n'
         'industry: Tech\n'
         "departments: ['R&D']"}


## 6. Mixed behavior in one table

One table can mix:

- scalar property inference
- explicit property overrides
- standard text-to-entity conversion
- explicit list-valued relation columns
- explicit foreign keys

This lets you model practical tabular sources without forcing strict first normal form before extraction.

In [11]:
projects_df = pd.DataFrame(
    [
        {
            "project_id": "P1",
            "dept_id": "D10",
            "budget_usd": 500000,
            "tags": "ml",
            "status_text": "In delivery",
        },
        {
            "project_id": "P2",
            "dept_id": "D20",
            "budget_usd": 1200000,
            "tags": "analytics, finance",
            "status_text": "Planned",
        },
    ]
)

documents, transformed_nodes = run_extractor(
    TabularTableSpec(
        name="departments",
        primary_key="dept_id",
        dataframe=departments_df,
    ),
    TabularTableSpec(
        name="projects",
        primary_key="project_id",
        dataframe=projects_df,
        property_columns=["status_text"],
        foreign_keys=[
            ForeignKeySpec(
                source_columns="dept_id",
                target_table="departments",
                target_columns="dept_id",
                relation_label="OWNED_BY_DEPARTMENT",
            )
        ],
        relation_columns=[
            RelationColumnSpec(
                column="tags",
                relation_label="HAS_TAG",
                target_label="Tag",
                split_separator=",",
            )
        ],
    ),
)

project_nodes = [
    node for node in transformed_nodes if node.metadata["ogre_kg_table_name"] == "projects"
]
pprint(graph_view(project_nodes[1]))

{'entities': [{'label': 'Projects',
               'name': 'projects:project_id=P2',
               'properties': {'budget_usd': 1200000,
                              'project_id': 'P2',
                              'status_text': 'Planned'}},
              {'label': 'Departments',
               'name': 'departments:dept_id=D20',
               'properties': {'dept_id': 'D20'}},
              {'label': 'Tag',
               'name': 'Tag:analytics',
               'properties': {'source_column': 'tags', 'value': 'analytics'}},
              {'label': 'Tag',
               'name': 'Tag:finance',
               'properties': {'source_column': 'tags', 'value': 'finance'}}],
 'relations': [{'label': 'OWNED_BY_DEPARTMENT',
                'properties': {'source_columns': ['dept_id'],
                               'target_columns': ['dept_id'],
                               'target_table': 'departments'},
                'source_id': 'projects:project_id=P2',
                'target_id':

## Next steps

From here, the transformed documents can be passed into a `PropertyGraphIndex` pipeline exactly like other OGRE KG extractors. The important difference is that the graph structure is derived deterministically from table specs instead of being inferred by an LLM.